# The question and the data

Each month an editor at FlyRank can review about 50 pages across a portfolio of client sites. This project ranks those pages. The 50 the editor opens first should be the ones about to lose search impressions.

Every number here comes from `reports/results.json`. `python -m bleedtracker reproduce` builds that file from the warehouse and checks it against my original results.

In [1]:
import pandas as pd

from bleedtracker.results import load_results

results = load_results()

## The question

At the close of a month, can the search signals known that day rank a client portfolio's pages so that the 50 an editor opens first are the ones about to lose impressions?

## The decision it supports

One editor, once a month, has room for about 50 pages. Today the shortlist is picked by hand. This work gives the editor an ordered list of 50 pages with a reason next to each one.

The editor opens a page and checks it. Then they rewrite the title and refresh the content, check the tracking first, or drop the item. Every item is a suggestion to look. Nothing on the list approves a rewrite, a merge or a removal on its own.

## The cost of a wrong call

A page that turns out fine costs the editor an afternoon. That time can be made up. A page that slides while nobody looks costs a month of search demand. Traffic lost to a competitor rarely comes back by itself.

The two mistakes don't weigh the same. I fixed precision@50 as the metric before any model existed. The editor needs the 50 pages they open to be worth opening.

## Why a ranking

In [2]:
envelope = results["decision_envelope"]
pd.Series(
    {
        "decision date": envelope["decision_date"],
        "pages I could score": f"{envelope['scorable_pages']:,}",
        "clients": envelope["clients"],
        "impressions in May 2026": f"{envelope['impressions_in_feature_month']:,}",
        "editor's capacity": envelope["editor_capacity"],
        "capacity as a share of the pages": f"{envelope['capacity_share_of_portfolio']:.3%}",
    },
    name="the sealed decision",
).to_frame()

,the sealed decision
decision date,2026-05-31
pages I could score,"82,570"
clients,24
impressions in May 2026,"202,738,833"
editor's capacity,50
capacity as a share of the pages,0.061%


In [3]:
pd.Series(envelope["base_rate_by_open_pair"], name="share of pages that declined").to_frame()

,share of pages that declined
2026-01 to 2026-02,0.2512
2026-02 to 2026-03,0.2290
2026-03 to 2026-04,0.5378
2026-04 to 2026-05,0.5377


Fifty slots cover 0.06% of the 82,570 pages I could score on 31 May 2026. In the four open months, between 22.9% and 53.8% of pages lost more than a fifth of their impressions the month after. A list of every declining page would run to tens of thousands of rows. The editor needs 50 of them, in order. That makes it a ranking problem.

## The data

The data is FlyRank's internship warehouse on Hugging Face, build v20260703. It covers 27 January 2025 to 30 June 2026 and needs an approved request to read. DuckDB reads only the months and columns a query uses. Nothing is downloaded.

In [4]:
data = results["data"]
pd.Series(data["release_row_counts"], name="rows").map("{:,}".format).to_frame()

,rows
dim_clients,104
dim_content,"519,606"
fact_content_query_90d,"2,414,248"
fact_content_daily_performance,"78,835,655"
fact_content_daily_performance_sample,"11,694,072"


The four tables of the star schema hold 81,769,613 rows. The daily fact table alone holds 78,835,655. When I say 79 million rows, I mean that table. It's the only one my features and labels come from. The `_sample` table is a copy of June 2026. It counts in neither total.

The daily table has one row per day, client and page. A check for duplicates in May 2026 found none.

## The slice

A row counts only when its search data is flagged available. A client counts when its search history starts on or before 1 January 2026. Each feature month is then a full month of history. A page counts when it had at least 100 impressions in the feature month. Below that, a 20% drop is noise.

These rules apply to the feature month only. A page that disappears the next month counts as a decline.

In [5]:
profile = data["feature_month_profile"]
pd.DataFrame(
    {
        "clients": [
            data["clients_total"],
            data["clients_eligible"],
            int(profile["clients"]),
            data["clients_in_sealed_slice"],
        ],
        "pages": [
            "",
            "",
            f"{data['page_rows_in_feature_month']:,}",
            f"{data['pages_scored_sealed']:,}",
        ],
    },
    index=[
        "in the release",
        "history from before 2026",
        "reporting search data in May 2026",
        "in the slice in May 2026",
    ],
)

,clients,pages
in the release,104,
history from before 2026,40,
reporting search data in May 2026,66,"237,910"
in the slice in May 2026,24,"82,570"


In May 2026, 237,910 pages had search data. 82,570 of them belong to eligible clients and cleared the 100-impression floor. That's 34.7%. The slice says nothing about the other 155,340 pages.

The slice is concentrated. The largest client holds 29.0% of the sealed month's pages. The top three hold 64.3%.

## The windows

The feature window is one calendar month. It closes on its last day. The label window is the whole next month. The two touch and never overlap. There are five month pairs, from January to February through May to June 2026. June 2026 stayed sealed until the end of the analysis.

## What I left out

- Anything from the label month. It comes from after the decision.
- Every analytics column. Which clients have analytics depends on what they bought. In May 2026 only 6.3% of daily rows had analytics flagged available. 20.7% had no flag at all.
- The 90-day query table. Its window overlaps every label window.
- The hashed ids. They only group and split pages. They're never model inputs.
- FlyRank's own scores and flags. They are another system's decisions.
- Update and optimization dates. The content table is a snapshot taken on 3 July 2026. 51.1% of the scored pages carry an update date after 31 May 2026.